# 02 — Revised Sepsis Experiment

**Revised protocol** comparing five causal methods across independent outer seeds:

| Method | Description |
|--------|-------------|
| `GLOBAL_SENTINEL` | One pooled model, sentinel encoding only |
| `GLOBAL_MISSINGNESS` | One pooled model + structural-absence indicators |
| `GLOBAL_MISSINGNESS_CDV` | One pooled model + absence indicators + CDV identity |
| `CDV_SEPARATE` | Separate model per retained CDV (proposed method) |
| `MATCHED_RANDOM_PARTITIONS` | Placebo: random partitions matching CDV sizes |

For each method, **all learners** are evaluated:
DR_RF · S_RF · S_Linear · T_RF · X_RF · Double_ML

**Primary learner for paper tables: DR_RF** (pre-specified, not selected by test performance).

Saves a checkpoint after every seed; re-running resumes from the last completed seed.

## 1. Setup

In [1]:
import sys, os
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
REVISED_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
for p in [PROJECT_ROOT, REVISED_ROOT]:
    if p not in sys.path:
        sys.path.insert(0, p)
os.chdir(PROJECT_ROOT)

import numpy as np
import pandas as pd
import torch
import warnings
warnings.filterwarnings('ignore')

# Load config
sys.path.insert(0, os.path.join(REVISED_ROOT, 'sepsis'))
import config as CFG

# Load helpers
from helpers.runner import run_outer_seed_loop_sepsis, load_checkpoint

print('Setup complete.')
print(f'N_OUTER_SEEDS        = {CFG.N_OUTER_SEEDS}')
print(f'TRAIN_PROP           = {CFG.TRAIN_PROP}')
print(f'CDV_COVERAGE_THRESHOLD= {CFG.CDV_COVERAGE_THRESHOLD}')
print(f'N_RANDOM_PERMUTATIONS = {CFG.N_RANDOM_PERMUTATIONS}')
print(f'DR_FINAL_MODEL        = {CFG.DR_FINAL_MODEL}')

Setup complete.
N_OUTER_SEEDS        = 20
TRAIN_PROP           = 0.7
CDV_COVERAGE_THRESHOLD= 0.9
N_RANDOM_PERMUTATIONS = 15
DR_FINAL_MODEL        = linear


## 2. Pre-run Checks — Verify Artifacts Exist

In [2]:
assert os.path.exists(CFG.DATASET_PATH), (
    f'sepsis_cases.csv not found at {CFG.DATASET_PATH}.\n'
    'Run 00_data_preparation.ipynb first.'
)
assert os.path.exists(CFG.REALCAUSE_CHECKPOINT), (
    f'RealCause checkpoint not found at {CFG.REALCAUSE_CHECKPOINT}.\n'
    'Run 01_realcause_training.ipynb first.'
)

print(f'Dataset:    {CFG.DATASET_PATH}  ✓')
print(f'Checkpoint: {CFG.REALCAUSE_CHECKPOINT}  ✓')

Dataset:    revised_experiment/sepsis/artifacts/sepsis_cases.csv  ✓
Checkpoint: revised_experiment/sepsis/artifacts/realcause_model/medium_seed_420/model.pt  ✓


## 3. Load Fixed RealCause DGP (once — never retrained)

In [3]:
from data.sepsis import load_sepsis
from models import TarNet, preprocess, MLPParams
from models import distributions
from cdv_utils.generator_validation import (
    setup_model_architectures, setup_training_parameters,
    setup_outcome_distribution, estimate_sigmoid_flow_cate
)

# Load raw covariates (needed to reconstruct the TarNet with correct dimensions)
df_cases_raw = pd.read_csv(CFG.DATASET_PATH)
w_cols = [c for c in df_cases_raw.columns if c not in ['t', 'y', 'y0', 'y1', 'ite', 'variant']]

sepsis_data = load_sepsis(data_format='numpy', dataroot=CFG.ARTIFACTS_DIR)
w_source, t_source, y_source = sepsis_data

# Reconstruct TarNet architecture (identical to training)
arch = setup_model_architectures()[0]
mlp_params = MLPParams(
    n_hidden_layers=arch['n_hidden_layers'],
    dim_h=arch['dim_h'],
    activation=arch['activation'],
)
network_params = {
    'mlp_params_w':    mlp_params,
    'mlp_params_t_w':  mlp_params,
    'mlp_params_y0_w': mlp_params,
    'mlp_params_y1_w': mlp_params,
}

realcause_model = TarNet(
    w_source, t_source, y_source,
    training_params=setup_training_parameters(),
    network_params=network_params,
    binary_treatment=True,
    outcome_distribution=setup_outcome_distribution(),
    outcome_min=0, outcome_max=1,
    train_prop=0.5, val_prop=0.1, test_prop=0.4,
    seed=420,
    early_stop=True, patience=500,
    ignore_w=False, grad_norm=float('inf'),
    w_transform=preprocess.Standardize,
    y_transform=preprocess.Normalize,
    savepath=CFG.REALCAUSE_CHECKPOINT,
)

# Load saved weights
checkpoint_states = torch.load(CFG.REALCAUSE_CHECKPOINT, map_location='cpu')
for network, state in zip(realcause_model.networks, checkpoint_states):
    network.load_state_dict(state)
    network.eval()

# Hash check: verify model is identical across all seeds
model_hash = hash(tuple(
    p.sum().item()
    for net in realcause_model.networks
    for p in net.parameters()
))
print(f'RealCause model loaded.  Weight hash: {model_hash}')
print('This hash must be identical for all seeds. Save it here for reference.')

# True CATE function using oracle inverse-CDF quadrature
true_cate_fn = lambda w_array: estimate_sigmoid_flow_cate(
    realcause_model, w_array, n_quantiles=128, batch_rows=16
)

print(f'\nw_cols ({len(w_cols)}): {w_cols[:5]} ...')

Loading sepsis dataset from revised_experiment/sepsis/artifacts\sepsis_cases.csv
n_train: 405	n_val: 81	n_test: 324
test_idxs:  (324,)
RealCause model loaded.  Weight hash: 4005009687056767623
This hash must be identical for all seeds. Save it here for reference.

w_cols (30): ['InfectionSuspected', 'DiagnosticBlood', 'DisfuncOrg', 'SIRSCritTachypnea', 'Hypotensie'] ...


## 4. Load Sepsis Cases DataFrame

In [4]:
# df_cases contains the original covariate values with NaN for structurally absent features.
# NaN → -1 happens inside run_single_seed_sepsis (via fillna).
df_cases = df_cases_raw[w_cols].copy()  # keep NaN as-is; runner applies fillna
print(f'Cases loaded: {len(df_cases)}')
print(f'NaN count: {df_cases.isna().sum().sum()} (structural absence)')

Cases loaded: 810
NaN count: 3352 (structural absence)


## 5. Build Config Dict

In [5]:
config = {
    'TRAIN_PROP':             CFG.TRAIN_PROP,
    'TEST_PROP':              CFG.TEST_PROP,
    'CDV_COVERAGE_THRESHOLD': CFG.CDV_COVERAGE_THRESHOLD,
    'CDV_N_MIN':              CFG.CDV_N_MIN,
    'CDV_MIN_ARM_SIZE':       CFG.CDV_MIN_ARM_SIZE,
    'OVERLAP_LO':             CFG.OVERLAP_LO,
    'OVERLAP_HI':             CFG.OVERLAP_HI,
    'OVERLAP_MIN_FRACTION':   CFG.OVERLAP_MIN_FRACTION,
    'RF_N_ESTIMATORS':        CFG.RF_N_ESTIMATORS,
    'DR_FINAL_MODEL':         CFG.DR_FINAL_MODEL,
    'DR_CV':                  CFG.DR_CV,
    'N_RANDOM_PERMUTATIONS':  CFG.N_RANDOM_PERMUTATIONS,
    'N_CDV_BOOTSTRAP':        CFG.N_CDV_BOOTSTRAP,
    'N_ORACLE_CV_FOLDS':      CFG.N_ORACLE_CV_FOLDS,
    'SENTINEL_VALUE':         CFG.SENTINEL_VALUE,
    'POSITIVITY_CLIP':        CFG.POSITIVITY_CLIP,
}

os.makedirs(CFG.ARTIFACTS_DIR, exist_ok=True)
os.makedirs(CFG.PLOTS_DIR, exist_ok=True)
print('Config dict built.')
print(f'POSITIVITY_CLIP  = {CFG.POSITIVITY_CLIP}  (propensities clipped to [{CFG.POSITIVITY_CLIP:.2f}, {1-CFG.POSITIVITY_CLIP:.2f}])')
print(f'DR_FINAL_MODEL   = {CFG.DR_FINAL_MODEL}')
print(f'DR_CV            = {CFG.DR_CV}  (cross-fitting folds; 5 = standard, 1 = none)')
print(f'N_CDV_BOOTSTRAP  = {CFG.N_CDV_BOOTSTRAP}  (CDV_SEPARATE ensemble resamples)')


Config dict built.
POSITIVITY_CLIP  = 0.0  (propensities clipped to [0.00, 1.00])
DR_FINAL_MODEL   = linear
DR_CV            = 3  (cross-fitting folds; 5 = standard, 1 = none)
N_CDV_BOOTSTRAP  = 1  (CDV_SEPARATE ensemble resamples)


## 6. Quick Debug Run (2 seeds, verify no crashes)

Set `RUN_DEBUG = True` to test with 2 seeds before the full run.

In [6]:
RUN_DEBUG = False        # set True for a 2-seed smoke test
FORCE_RERUN_DEBUG = True  # delete stale debug checkpoint before running

if RUN_DEBUG:
    debug_config = dict(config)
    debug_config['N_RANDOM_PERMUTATIONS'] = 2
    debug_config['N_CDV_BOOTSTRAP']       = 2  # keep debug fast
    debug_config['N_ORACLE_CV_FOLDS']     = 1
    debug_ckpt = CFG.CHECKPOINT_PATH.replace('.pkl', '_debug.pkl')

    if FORCE_RERUN_DEBUG and os.path.exists(debug_ckpt):
        os.remove(debug_ckpt)
        print(f'Removed stale debug checkpoint: {debug_ckpt}')

    from helpers.runner import run_outer_seed_loop_sepsis
    debug_results = run_outer_seed_loop_sepsis(
        outer_seeds=[0, 1],
        config=debug_config,
        realcause_model=realcause_model,
        df_cases=df_cases,
        w_cols=w_cols,
        true_cate_fn=true_cate_fn,
        checkpoint_path=debug_ckpt,
    )
    print(f'Debug run complete: {len(debug_results)} seeds.')
    # Show metrics for first seed
    first = debug_results[0]
    print('\nSample metrics (seed 0, DR_RF):')
    for method in ['GLOBAL_SENTINEL', 'CDV_SEPARATE']:
        m = first['metrics'].get(method, {}).get('DR_RF', {})
        ate  = m.get('ate_mse',  float('nan'))
        cate = m.get('cate_mse', float('nan'))
        print(f'  {method}: ATE MSE={ate:.6f}, CATE MSE={cate:.6f}')
    print(f'\n  Retained CDVs: {list(first["retained_cdv_info"].keys())}')
    print(f'  OTHER% (train): {first["pct_other_train"]:.1f}')
else:
    print('Debug run skipped. Set RUN_DEBUG=True to run a smoke test first.')


Debug run skipped. Set RUN_DEBUG=True to run a smoke test first.


## 7. Full Experiment — Outer Seed Loop

In [7]:
FORCE_RERUN_FULL = False  # delete stale full-run checkpoint before running

if FORCE_RERUN_FULL and os.path.exists(CFG.CHECKPOINT_PATH):
    os.remove(CFG.CHECKPOINT_PATH)
    print(f'Removed stale checkpoint: {CFG.CHECKPOINT_PATH}')

print(f'Starting full experiment: {CFG.N_OUTER_SEEDS} seeds.')
print(f'Checkpoint: {CFG.CHECKPOINT_PATH}')
print('Resumes automatically from last completed seed if re-run.\n')

results_by_seed = run_outer_seed_loop_sepsis(
    outer_seeds=CFG.OUTER_SEEDS,
    config=config,
    realcause_model=realcause_model,
    df_cases=df_cases,
    w_cols=w_cols,
    true_cate_fn=true_cate_fn,
    checkpoint_path=CFG.CHECKPOINT_PATH,
)

print(f'\nExperiment complete! {len(results_by_seed)} seeds processed.')

Starting full experiment: 20 seeds.
Checkpoint: revised_experiment/sepsis/artifacts/results_checkpoint.pkl
Resumes automatically from last completed seed if re-run.

Loaded 0 completed seeds from checkpoint.
[Seed 0] Split: 567 train / 243 test (treatment rates: 0.109 / 0.107)

[Seed 0] Starting core runner...
[Seed 0] Discovered 2 CDV patterns.
[Seed 0] CDV_2 FAILED retention: ['n_treated=0 < CDV_MIN_ARM_SIZE=2', 'OptionA: mean_treatment_rate=0.000 < lo=0.05', 'OptionB: overlap_fraction=0.080 < min_fraction=0.2']
[Seed 0] Retained 1 CDVs; 1 sent to OTHER due to retention failure.
[Seed 0] Computing oracle CATE via true_cate_fn...
[Seed 0] Running method: GLOBAL_SENTINEL...
[Seed 0] Running method: GLOBAL_MISSINGNESS...
[Seed 0] Running method: GLOBAL_MISSINGNESS_CDV...
[Seed 0] Running method: CDV_SEPARATE...
[Seed 0] Running method: MATCHED_RANDOM_PARTITIONS...
[Seed 0] Running oracle selection...
[Seed 0] ✓ Saved.
[Seed 1] Split: 567 train / 243 test (treatment rates: 0.086 / 0.086)

## 8. Summary

In [8]:
results_by_seed = load_checkpoint(CFG.CHECKPOINT_PATH)
completed = sorted(results_by_seed.keys())
print(f'Completed seeds ({len(completed)}): {completed}')

# Quick per-seed violation summary
print('\nCDV violation summary:')
for seed in completed:
    viols = results_by_seed[seed].get('violations_log', {})
    n_retained = len(results_by_seed[seed].get('retained_cdv_info', {}))
    pct_other = results_by_seed[seed].get('pct_other_train', np.nan)
    print(f'  Seed {seed}: retained={n_retained}, failed={len(viols)}, OTHER%={pct_other:.1f}')

# Quick metric preview (DR_RF, GLOBAL_SENTINEL vs CDV_SEPARATE)
print('\nQuick metric preview (DR_RF):')
print(f'{"Seed":<6} {"GS_ATE_MSE":<15} {"CDV_ATE_MSE":<15}')
for seed in completed[:5]:
    gs  = results_by_seed[seed]['metrics'].get('GLOBAL_SENTINEL', {}).get('DR_RF', {}).get('ate_mse', np.nan)
    cdv = results_by_seed[seed]['metrics'].get('CDV_SEPARATE', {}).get('DR_RF', {}).get('ate_mse', np.nan)
    print(f'{seed:<6} {gs:<15.5f} {cdv:<15.5f}')

Completed seeds (20): [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]

CDV violation summary:
  Seed 0: retained=1, failed=1, OTHER%=18.2
  Seed 1: retained=1, failed=1, OTHER%=20.3
  Seed 2: retained=1, failed=1, OTHER%=18.9
  Seed 3: retained=1, failed=1, OTHER%=18.5
  Seed 4: retained=1, failed=1, OTHER%=18.0
  Seed 5: retained=1, failed=1, OTHER%=18.3
  Seed 6: retained=1, failed=2, OTHER%=18.3
  Seed 7: retained=1, failed=2, OTHER%=19.4
  Seed 8: retained=1, failed=1, OTHER%=16.6
  Seed 9: retained=1, failed=1, OTHER%=18.0
  Seed 10: retained=1, failed=1, OTHER%=17.8
  Seed 11: retained=1, failed=1, OTHER%=18.0
  Seed 12: retained=1, failed=1, OTHER%=17.1
  Seed 13: retained=1, failed=2, OTHER%=18.3
  Seed 14: retained=1, failed=1, OTHER%=17.1
  Seed 15: retained=1, failed=2, OTHER%=20.1
  Seed 16: retained=1, failed=1, OTHER%=19.2
  Seed 17: retained=1, failed=1, OTHER%=16.2
  Seed 18: retained=1, failed=1, OTHER%=19.9
  Seed 19: retained=1, failed=1, OTHER